# TurboQuant: Near-Optimal Vector Quantization

Интерактивный туториал с визуализациями алгоритма TurboQuant.

**Ключевые идеи:**
1. Случайный поворот → координаты становятся почти независимыми
2. Lloyd-Max квантизатор для каждой координаты
3. QJL остаток для устранения смещения скалярного произведения

[arXiv:2504.19874v1](https://arxiv.org/html/2504.19874v1)

## Версии

TurboQuant доступен в двух версиях:
- **Pure Python** (`from turboquant import TurboQuantMSE`) — оригинальная реализация на NumPy/SciPy
- **Rust bindings** (`from turboquant_rs import TurboQuantMse`) — высокая производительность через PyO3/maturin

В этом туториале используется Pure Python версия для наглядности. Rust API аналогичен.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from turboquant import TurboQuantMSE, TurboQuantProd, QJL, _lloyd_max_gaussian, _lloyd_max_beta_sphere

np.random.seed(42)
rng = np.random.default_rng(42)

# Настройки визуализации
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Проверка доступности Rust bindings
try:
    from turboquant_rs import TurboQuantMse as TurboQuantMse_Rust
    HAS_RUST = True
    print('✓ Rust bindings available')
except ImportError:
    HAS_RUST = False
    print('✗ Rust bindings not installed (run: pip install maturin && maturin develop --release)')

## 1. Маргинальное распределение координат сферы

После случайного поворота координаты вектора на сфере S^{d-1} имеют распределение:

$$f_X(x) = \frac{\Gamma(d/2)}{\sqrt{\pi}\,\Gamma((d-1)/2)} (1 - x^2)^{(d-3)/2}$$

In [ ]:
from scipy.special import gamma as gamma_fn

def sphere_pdf(x, d):
    coeff = gamma_fn(d / 2) / (np.sqrt(np.pi) * gamma_fn((d - 1) / 2))
    alpha = (d - 3) / 2
    return coeff * np.maximum(1 - x**2, 0)**alpha

# Визуализация для разных d
x = np.linspace(-1, 1, 500)
fig, ax = plt.subplots()
for d in [2, 4, 8, 16, 64, 256]:
    y = sphere_pdf(x, d)
    ax.plot(x, y, label=f'd={d}')

ax.set_xlabel('x')
ax.set_ylabel('f_X(x)')
ax.set_title('Маргинальное распределение координат сферы S^{d-1}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

При d → ∞ распределение сходится к N(0, 1/d).

## 2. Lloyd-Max квантизатор

Оптимальные центроиды для заданного распределения.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (d, n_centroids) in zip(axes, [(8, 4), (64, 8), (256, 16)]):
    centroids = _lloyd_max_beta_sphere(n_centroids, d)
    
    x_plot = np.linspace(-1, 1, 500)
    y_plot = sphere_pdf(x_plot, d)
    ax.plot(x_plot, y_plot, 'b-', alpha=0.5, label='PDF')
    
    # Вертикальные линии для центроидов
    for c in centroids:
        ax.axvline(c, color='r', linestyle='--', alpha=0.7)
    
    ax.set_title(f'd={d}, k={n_centroids}')
    ax.set_xlabel('x')
    ax.grid(True, alpha=0.3)

fig.suptitle('Lloyd-Max центроиды для маргинала сферы', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Эффект случайного поворота

Без поворота координаты коррелированы → квантование неэффективно.
После поворота координаты почти независимы → скалярное квантование работает отлично.

In [ ]:
d = 16  # Малая размерность для наглядности
n = 5000

# Один вектор, много случайных поворотов
x0 = np.random.randn(d)
x0 /= np.linalg.norm(x0)

# Без поворота: координаты фиксированы
coords_no_rotation = np.tile(x0, (n, 1))

# С поворотом: QR случайной матрицы
coords_rotated = np.zeros((n, d))
for i in range(n):
    A = rng.standard_normal((d, d))
    Q, _ = np.linalg.qr(A)
    Q *= np.sign(np.diag(np.linalg.qr(A)[1]))
    coords_rotated[i] = Q @ x0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма первой координаты
axes[0].hist(coords_no_rotation[:, 0], bins=30, alpha=0.7, density=True, label='Без поворота')
axes[0].set_title('Первая координата (без поворота)')
axes[0].set_xlabel('x₁')
axes[0].grid(True, alpha=0.3)

axes[1].hist(coords_rotated[:, 0], bins=30, alpha=0.7, density=True, label='С поворотом', color='orange')
x_theory = np.linspace(-1, 1, 200)
axes[1].plot(x_theory, sphere_pdf(x_theory, d), 'r-', lw=2, label='Теория')
axes[1].set_title('Первая координата (случайный поворот)')
axes[1].set_xlabel('x₁')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. MSE vs количество бит

Теория: MSE убывает как ~1/4^b с каждым дополнительным битом.

In [ ]:
d, n = 256, 2000
X = rng.standard_normal((n, d))
X /= np.linalg.norm(X, axis=1, keepdims=True)

bits_range = [1, 2, 3, 4, 5, 6, 8]
mse_values = []
setup_times = []

import time
for b in bits_range:
    t0 = time.perf_counter()
    q = TurboQuantMSE(d, b, seed=42)
    setup_t = time.perf_counter() - t0
    mse = q.mse(X)
    mse_values.append(mse)
    setup_times.append(setup_t)

fig, ax = plt.subplots()
ax.semilogy(bits_range, mse_values, 'bo-', linewidth=2, markersize=8)

# Теоретическая кривая 1/4^b (нормированная)
scale = mse_values[0] * 4**bits_range[0]
theoretical = [scale / 4**b for b in bits_range]
ax.semilogy(bits_range, theoretical, 'r--', alpha=0.5, label='~1/4^b')

ax.set_xlabel('Бит на координату (b)')
ax.set_ylabel('MSE')
ax.set_title('MSE vs количество бит (d=256)')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Фактический decay per bit:')
for i in range(1, len(bits_range)):
    db = bits_range[i] - bits_range[i-1]
    decay = (mse_values[i-1] / mse_values[i]) ** (1/db)
    print(f'  b={bits_range[i-1]}→{bits_range[i]}: {decay:.1f}×/bit')

## 5. Несмещённость оценки скалярного произведения

TurboQuantProd обеспечивает E[⟨y, x̃⟩] = ⟨y, x⟩

> **Примечание:** Pure Python API возвращает кортеж `(mse_idx, qjl_signs, res_norms)`, тогда как Rust API возвращает объект `QuantizedProd` с атрибутами `mse_indices`, `qjl_signs`, `residual_norm`.

In [ ]:
d, n = 128, 1000
X = rng.standard_normal((n, d))
X /= np.linalg.norm(X, axis=1, keepdims=True)
y = rng.standard_normal(d)
y /= np.linalg.norm(y)

true_ips = X @ y

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, b in zip(axes, [2, 4, 8]):
    q = TurboQuantProd(d, b, seed=42)
    mse_idx, qjl_signs, res_norms = q.encode(X)
    est_ips = np.array([
        q.inner_product_estimate(y, mse_idx[i], qjl_signs[i], res_norms[i])
        for i in range(n)
    ])
    
    bias = np.mean(est_ips - true_ips)
    rmse = np.sqrt(np.mean((est_ips - true_ips)**2))
    
    ax.scatter(true_ips, est_ips, alpha=0.3, s=5)
    ax.plot([true_ips.min(), true_ips.max()], [true_ips.min(), true_ips.max()], 
            'r--', alpha=0.5, label='Идеально')
    ax.set_xlabel('True IP')
    ax.set_ylabel('Estimated IP')
    ax.set_title(f'b={b}  bias={bias:+.5f}  RMSE={rmse:.5f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('Несмещённость оценки скалярного произведения', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Сжатие: качество vs экономия памяти

In [ ]:
d = 256
raw_bytes = d * 8  # float64

print(f'{'Метод':<12} {'b':>3} {'Байт/вектор':>14} {'Сжатие':>10} {'MSE':>10}')
print('-' * 55)

for b in [1, 2, 3, 4, 5, 6, 8]:
    q = TurboQuantMSE(d, b, seed=42)
    mse = q.mse(X[:500])
    quant_bytes = (b * d) / 8
    ratio = raw_bytes / quant_bytes
    print(f'{'MSE':<12} {b:>3} {quant_bytes:>14.0f} {ratio:>9.1f}x {mse:>10.6f}')

for b in [2, 4, 8]:
    q = TurboQuantProd(d, b, seed=42)
    prod_bytes = q.bits_per_vector() / 8
    ratio = raw_bytes / prod_bytes
    print(f'{'Prod':<12} {b:>3} {prod_bytes:>14.0f} {ratio:>9.1f}x {'—':>10}')

## 7. Быстрый старт

### MSE-квантование (Pure Python)

In [ ]:
from turboquant import TurboQuantMSE

d, b = 256, 4
q = TurboQuantMSE(d=d, b=b, seed=42)

# Один вектор
x = rng.standard_normal(d)
x /= np.linalg.norm(x)

idx   = q.encode(x)      # (256,) uint16 — индексы центроидов
x_hat = q.decode(idx)    # (256,) float64 — реконструкция

print(f'Размер индексов: {idx.nbytes} байт (было {x.nbytes} байт)')
print(f'Сжатие: {x.nbytes / idx.nbytes:.1f}x')
print(f'MSE: {np.mean((x - x_hat)**2):.6f}')

### MSE-квантование (Rust bindings)

In [ ]:
if HAS_RUST:
    from turboquant_rs import TurboQuantMse
    
    d, b = 256, 4
    q = TurboQuantMse(d=d, b=b, seed=42)
    
    # Один вектор (list[float] вместо numpy)
    x = rng.standard_normal(d).tolist()
    x_norm = [v / np.linalg.norm(x) for v in x]
    
    idx   = q.encode(x_norm)   # List[int] — индексы центроидов
    x_hat = q.decode(idx)      # List[float] — реконструкция
    
    print(f'✓ Rust bindings работают')
    print(f'  Свойства: d={q.d}, b={q.b}, n_centroids={q.n_centroids}')
    print(f'  Сжатие: {d * 64 / (len(idx) * 16):.1f}x')  # float64→uint16
    print(f'  MSE: {q.mse(x_norm):.6f}')
else:
    print('Rust bindings не установлены')

### Несмещённое скалярное произведение (Pure Python)

In [ ]:
from turboquant import TurboQuantProd

q = TurboQuantProd(d=256, b=4, seed=42)

# База векторов
X = rng.standard_normal(1000, 256)
X /= np.linalg.norm(X, axis=1, keepdims=True)
mse_idx, qjl_signs, res_norms = q.encode(X)

# Запрос
y = rng.standard_normal(256)
y /= np.linalg.norm(y)

# Оценка одного скалярного произведения
i = 42
ip_est = q.inner_product_estimate(y, mse_idx[i], qjl_signs[i], res_norms[i])
ip_true = np.dot(X[i], y)

print(f'Истинное IP: {ip_true:.6f}')
print(f'Оценка IP:   {ip_est:.6f}')
print(f'Ошибка:      {abs(ip_est - ip_true):.6f}')

### Несмещённое скалярное произведение (Rust bindings)

In [ ]:
if HAS_RUST:
    from turboquant_rs import TurboQuantProd
    
    q = TurboQuantProd(d=256, b=4, seed=42)
    
    # Один вектор
    x = rng.standard_normal(256)
    x /= np.linalg.norm(x)
    
    # encode возвращает QuantizedProd объект
    qv = q.encode(x.tolist())
    
    # Свойства QuantizedProd
    print(f'QuantizedProd: {qv}')
    print(f'  mse_indices: {len(qv.mse_indices)}')
    print(f'  qjl_signs: {len(qv.qjl_signs)}')
    print(f'  residual_norm: {qv.residual_norm:.6f}')
    
    # Оценка скалярного произведения
    y = rng.standard_normal(256)
    y /= np.linalg.norm(y)
    ip_est = q.inner_product_estimate(y.tolist(), qv)
    ip_true = np.dot(x, y)
    
    print(f'\nИстинное IP: {ip_true:.6f}')
    print(f'Оценка IP:   {ip_est:.6f}')
    print(f'Ошибка:      {abs(ip_est - ip_true):.6f}')
else:
    print('Rust bindings не установлены')

## 8. Сравнение производительности (Pure Python vs Rust)

In [ ]:
if HAS_RUST:
    from turboquant import TurboQuantMSE as TurboQuantMSE_Py
    from turboquant_rs import TurboQuantMse as TurboQuantMse_Rs
    
    d, b, n = 256, 4, 1000
    X = rng.standard_normal((n, d))
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    
    # Pure Python
    q_py = TurboQuantMSE_Py(d, b, seed=42)
    t0 = time.perf_counter()
    _ = q_py.encode(X)
    t_py = time.perf_counter() - t0
    
    # Rust
    q_rs = TurboQuantMse_Rs(d, b, seed=42)
    X_list = X.tolist()
    t0 = time.perf_counter()
    _ = q_rs.encode_batch(X_list) if hasattr(q_rs, 'encode_batch') else [q_rs.encode(row) for row in X_list]
    t_rs = time.perf_counter() - t0
    
    print(f'Pure Python encode: {t_py*1000:.1f}ms')
    print(f'Rust encode:        {t_rs*1000:.1f}ms')
    print(f'Speedup:            {t_py/t_rs:.1f}×')
else:
    print('Rust bindings не установлены (для сравнения запустите: maturin develop --release)')